In [13]:
# Monkey-patch to get live output from python file into this notebook
import subprocess
import sys

_original_run = subprocess.run

def notebook_run(args, **kwargs):
    process = subprocess.Popen(
        [str(arg) for arg in args],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )

    for line in process.stdout:
        print(line, end="")
        sys.stdout.flush()

    return_code = process.wait()
    if return_code:
        raise subprocess.CalledProcessError(return_code, args)

    return subprocess.CompletedProcess(args, return_code)

subprocess.run = notebook_run


In [27]:
%load_ext autoreload
%autoreload 2

import importlib
import src.combine
importlib.reload(src.combine)

from src.combine import run_colmap
import open3d as o3d


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [28]:
# run colmap
runit = run_colmap(image_dir="aloi_data/nerf_llff_data/fern/images", output_dir="outputs/demo")
# runit.run()   # this takes a long time to run, just use old outputs for testing changes to the run_colmap object

In [32]:
# visualize
ply_path = "outputs/demo/dense/fused.ply"

pcd = o3d.io.read_point_cloud(ply_path)

print(pcd)
print("Number of points:", len(pcd.points))

o3d.visualization.draw_geometries([pcd])

PointCloud with 1787638 points.
Number of points: 1787638


In [ ]:
# remove noise
clean_pcd, noise_idx = runit.remove_noise()
# clean_pcd.paint_uniform_color([1, 0.706, 0])

# select noise
noise = pcd.select_by_index(noise_idx, invert=True)
noise.paint_uniform_color([1, 0, 0])

# visualize
o3d.visualization.draw_geometries([clean_pcd])

In [33]:
subsampled_pcd = runit.adaptive_subsample()
o3d.visualization.draw_geometries([clean_pcd])
print("Number of points:", len(subsampled_pcd.points))
print(f"Subsampling reduced number of points by {100 * (1 - len(subsampled_pcd.points) / len(pcd.points))}%")

Number of points: 774631
Subsampling reduced number of points by 56.66734540214517%
